# 02 Databases and Caching

Connections live in `settings/connections.yml` and are opened by name: `framework` (the
project's `framework.db`), `study` (the shared `../../shared/study.sqlite`, with
`read_only: true`), and `local` (`local.sqlite`, ignored by git, for writes).

In [ ]:
import time

import framework as fw
import statsmodels.formula.api as smf

root = fw.scaffold()

## Connections

A relative SQLite path is resolved from the project root, so the `study` path works from
`notebooks/` too. `connections_list()` marks the default.

In [ ]:
fw.connections_list()

In [ ]:
fw.db_drivers_status()[["drivers", "installed", "install"]]

## Queries

`db_query()` opens, queries, closes, and returns a DataFrame. Mean systolic pressure by
arm and visit: the intervention arm falls away from control after visit 1.

In [ ]:
bp_by_visit = fw.db_query(
    """
    SELECT p.treatment, v.visit_number, COUNT(v.systolic_bp) AS n,
           ROUND(AVG(v.systolic_bp), 1) AS mean_systolic
    FROM visits v JOIN participants p USING (participant_id)
    GROUP BY p.treatment, v.visit_number
    """,
    "study",
)
bp_by_visit.pivot(index="visit_number", columns="treatment", values="mean_systolic")

Values go in `params`, with `?` placeholders for SQLite.

In [ ]:
fw.db_query(
    "SELECT test, unit, ROUND(AVG(value), 2) AS mean, COUNT(value) AS n "
    "FROM labs WHERE test IN (?, ?) GROUP BY test, unit",
    "study",
    params=["ldl", "crp"],
)

## A read-only connection

`study` has `read_only: true`, so it opens in SQLite's read-only mode and `db_execute()`
refuses it.

In [ ]:
try:
    fw.db_execute("DELETE FROM visits", "study")
except fw.FrameworkError as error:
    print(error)

## Writes and transactions

`db_execute()` runs one statement on `local`, commits, and returns the row count.

In [ ]:
fw.db_execute("DROP TABLE IF EXISTS bp_summary", "local")
fw.db_execute("CREATE TABLE bp_summary (treatment TEXT, visit INTEGER, mean REAL)", "local")
sum(
    fw.db_execute("INSERT INTO bp_summary VALUES (?, ?, ?)", "local",
                  params=[r.treatment, r.visit_number, r.mean_systolic])
    for r in bp_by_visit.itertuples()
)

`db_connect()` with `db_transaction()` commits a block together, or rolls all of it back.
The insert below is undone because the second statement fails.

In [ ]:
try:
    with fw.db_connect("local") as conn, fw.db_transaction(conn):
        cur = conn.cursor()
        cur.execute("INSERT INTO bp_summary VALUES ('placebo', 1, 0)")
        cur.execute("INSERT INTO missing_table VALUES (1)")
except fw.FrameworkError as error:
    print(error)

fw.db_query("SELECT COUNT(*) AS rows FROM bp_summary", "local")

`read_only=` overrides the setting for one call: `local` is writable, but opened with
`read_only=True` it refuses writes, as `study` does.

In [ ]:
with fw.db_connect("local", read_only=True) as conn:
    print("local read_only:", conn.read_only)
    try:
        with fw.db_transaction(conn):
            pass
    except fw.FrameworkError as error:
        print(error)

## Caching

`cache_remember()` computes a value once and returns the cached copy after that, across
sessions. The mixed model below has a random intercept per participant. Run the cell
twice and compare the times.

In [ ]:
bp = fw.data_read("inputs.intermediate.bp_visits")

def fit_bp_model():
    model = smf.mixedlm("systolic_bp ~ visit_number * treatment + age",
                        data=bp, groups=bp["participant_id"])
    return model.fit().params

start = time.perf_counter()
params = fw.cache_remember("bp_mixed_params", fit_bp_model)
print(f"{time.perf_counter() - start:.3f} s")
params.round(2)

As a decorator, with an expiry of 30 minutes (seconds, hours, days, and weeks work too):

In [ ]:
@fw.cache_remember("site_counts", expire_after="30 minutes")
def site_counts():
    return fw.db_query("SELECT site_name, COUNT(*) AS n FROM participant_summary "
                       "GROUP BY site_name", "study")

site_counts()

`depends_on` recomputes when an input changes: a catalog file, a file or folder, or a
parameter. The third call has a new cutoff, so it computes again.

In [ ]:
for cutoff in (140, 140, 130):
    share = fw.cache_remember(
        "share_high_bp",
        lambda: round(float((bp["systolic_bp"] >= cutoff).mean()), 3),
        depends_on=["inputs.intermediate.bp_visits", {"cutoff": cutoff}],
    )
    print(cutoff, share)

`cache()` and `cache_get()` store and read directly; `cache_list()` shows the records.

In [ ]:
fw.cache("n_participants", int(bp["participant_id"].nunique()))
print(fw.cache_get("n_participants"), fw.cache_get("nope", default="(none)"))
fw.cache_list()[["name", "expire_at", "status"]]

`cache_forget()` drops one value. `cache_flush()` drops all of them; it is commented out
so the next run shows a cache hit.

In [ ]:
fw.cache_forget("n_participants")
# fw.cache_flush()
fw.cache_list()["name"].tolist()